# 03 — Modeling: predicting stickiness

We train interpretable models to predict the **sticky** label (top ~20% popularity) from audio features. This does **not** claim audio *causes* popularity — marketing, artist reach, and timing confound the outcome.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import statsmodels.api as sm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import features
from src import modeling
from src import visuals

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
TBL_DIR = PROJECT_ROOT / "outputs" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "spotify_model_data.csv"
df = pd.read_csv(DATA_PATH)
df = df.reset_index(drop=True)
print(df.shape)


(2500, 16)


## 1. Feature matrix & train/test split (80/20, stratified when possible)


In [2]:
INCLUDE_GENRE = False

X, y = features.split_features_target(df, target_col="sticky", include_genre=INCLUDE_GENRE)
feature_names = list(X.columns)
assert X.notna().all().all(), "NaNs in feature matrix"

X_train, X_test, y_train, y_test = modeling.train_test_split_stratified(
    X, y, test_size=0.2, random_state=42
)

X_train_s, X_test_s, scaler = features.scale_train_test(X_train, X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Features:", feature_names)


Train: (2000, 10) Test: (500, 10)
Features: ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'duration_ms']


## 2. Logistic regression (baseline, standardized features)


In [3]:
log_reg = modeling.train_logistic_regression(X_train_s, y_train, class_weight='balanced')
log_metrics = modeling.evaluate_classifier(log_reg, X_test_s, y_test)
print("Logistic regression metrics:", log_metrics)

fig = visuals.plot_logistic_coefficients(
    log_reg, feature_names, save_path=FIG_DIR / "09_logistic_coefficients.png"
)
plt.show()

coef = log_reg.coef_.ravel()
order = np.argsort(coef)
print("Top negative (lower log-odds for sticky):", list(zip(np.array(feature_names)[order[:5]], coef[order[:5]])))
print("Top positive (higher log-odds for sticky):", list(zip(np.array(feature_names)[order[-5:]], coef[order[-5:]])))


Logistic regression metrics: {'accuracy': 0.552, 'precision': 0.2413793103448276, 'recall': 0.5384615384615384, 'f1': 0.3333333333333333, 'roc_auc': 0.5901806526806527}
Top negative (lower log-odds for sticky): [(np.str_('loudness'), np.float64(-0.11693670730531323)), (np.str_('instrumentalness'), np.float64(-0.06926614330654583)), (np.str_('acousticness'), np.float64(-0.0036197185221271246)), (np.str_('liveness'), np.float64(0.007821650396046636)), (np.str_('duration_ms'), np.float64(0.0259219462147822))]
Top positive (higher log-odds for sticky): [(np.str_('tempo'), np.float64(0.028346080532337215)), (np.str_('valence'), np.float64(0.07449621340540886)), (np.str_('speechiness'), np.float64(0.1322817574870122)), (np.str_('energy'), np.float64(0.2171634393589161)), (np.str_('danceability'), np.float64(0.26238838138374626))]


/var/folders/6l/d9f_7jpd3934kc_kmx1xbsbr0000gn/T/ipykernel_19065/3633410558.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Random forest


In [4]:
rf = modeling.train_random_forest(X_train, y_train, class_weight='balanced')
rf_metrics = modeling.evaluate_classifier(rf, X_test, y_test)
print("Random forest metrics:", rf_metrics)

importances = rf.feature_importances_
fig = visuals.plot_feature_importance(
    importances,
    feature_names,
    title="Random forest feature importance",
    save_path=FIG_DIR / "10_rf_feature_importance.png",
)
plt.show()

imp_order = np.argsort(importances)
print("Top 5 RF importance:", list(zip(np.array(feature_names)[imp_order[-5:]], importances[imp_order[-5:]])))


Random forest metrics: {'accuracy': 0.794, 'precision': 1.0, 'recall': 0.009615384615384616, 'f1': 0.01904761904761905, 'roc_auc': 0.546207264957265}
Top 5 RF importance: [(np.str_('valence'), np.float64(0.09839895848656957)), (np.str_('liveness'), np.float64(0.09854426733881468)), (np.str_('loudness'), np.float64(0.1012960151537586)), (np.str_('energy'), np.float64(0.10954966243012997)), (np.str_('danceability'), np.float64(0.11719163462100864))]


/var/folders/6l/d9f_7jpd3934kc_kmx1xbsbr0000gn/T/ipykernel_19065/1307460716.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Model comparison


In [5]:
comp = modeling.build_classification_report_df({
    "logistic_regression": log_metrics,
    "random_forest": rf_metrics,
})
display(comp.round(4))
comp.to_csv(TBL_DIR / "model_metrics.csv")


,accuracy,precision,recall,f1,roc_auc
model,,,,,
logistic_regression,0.552,0.2414,0.5385,0.3333,0.5902
random_forest,0.794,1.0000,0.0096,0.0190,0.5462


## 5. Confusion matrices


In [6]:
fig = visuals.plot_confusion_matrix_from_model(
    log_reg, X_test_s, y_test, save_path=FIG_DIR / "11_confusion_matrix_logistic.png"
)
plt.show()
fig = visuals.plot_confusion_matrix_from_model(
    rf, X_test, y_test, save_path=FIG_DIR / "12_confusion_matrix_rf.png"
)
plt.show()


/var/folders/6l/d9f_7jpd3934kc_kmx1xbsbr0000gn/T/ipykernel_19065/2486431012.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/6l/d9f_7jpd3934kc_kmx1xbsbr0000gn/T/ipykernel_19065/2486431012.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5b. ROC and precision–recall curves

ROC summarizes rank discrimination; precision–recall is often more informative under **class imbalance** (~20% sticky).


In [7]:
pos_lr = int(np.where(log_reg.classes_ == 1)[0][0])
y_score_lr = log_reg.predict_proba(X_test_s)[:, pos_lr]
pos_rf = int(np.where(rf.classes_ == 1)[0][0])
y_score_rf = rf.predict_proba(X_test)[:, pos_rf]

fig = visuals.plot_roc_comparison(
    y_test, y_score_lr, y_score_rf, save_path=FIG_DIR / "13_roc_curves.png"
)
plt.show()
fig = visuals.plot_precision_recall_comparison(
    y_test, y_score_lr, y_score_rf, save_path=FIG_DIR / "14_pr_curves.png"
)
plt.show()


/var/folders/6l/d9f_7jpd3934kc_kmx1xbsbr0000gn/T/ipykernel_19065/3185261516.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/6l/d9f_7jpd3934kc_kmx1xbsbr0000gn/T/ipykernel_19065/3185261516.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Optional: regression on `popularity_z`


In [8]:
if "popularity_z" not in df.columns:
    print("popularity_z missing — skip regression block.")
else:
    yz_train = df.loc[X_train.index, "popularity_z"]
    yz_test = df.loc[X_test.index, "popularity_z"]
    lin = modeling.train_linear_regression(X_train_s, yz_train)
    reg_metrics = modeling.evaluate_regression(lin, X_test_s, yz_test)
    print("Regression on popularity_z:", reg_metrics)


Regression on popularity_z: {'r2': 0.03643972673235807, 'mae': 0.7746375056401228, 'rmse': 0.938025568098252}


## 7. Optional: OLS on `popularity_z`


In [9]:
if "popularity_z" not in df.columns:
    print("Skip OLS.")
else:
    y_pop = df.loc[X_train.index, "popularity_z"]
    X_ols = sm.add_constant(X_train_s, has_constant="add")
    ols = sm.OLS(y_pop, X_ols).fit()
    print(ols.summary().tables[1])


                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.0050      0.022      0.229      0.819      -0.038       0.048
danceability         0.1906      0.022      8.634      0.000       0.147       0.234
energy               0.1215      0.022      5.510      0.000       0.078       0.165
valence              0.0297      0.022      1.347      0.178      -0.014       0.073
tempo                0.0115      0.022      0.521      0.602      -0.032       0.055
loudness            -0.0487      0.022     -2.207      0.027      -0.092      -0.005
speechiness          0.0279      0.022      1.263      0.207      -0.015       0.071
acousticness         0.0031      0.022      0.141      0.888      -0.040       0.047
instrumentalness    -0.0467      0.022     -2.115      0.035      -0.090      -0.003
liveness            -0.0040      0.022     -0.180      0.857     

## 8. Error analysis (logistic)


In [10]:
test_idx = X_test.index
meta_cols = [c for c in ("track_name", "artist_name", "genre", "popularity") if c in df.columns]

y_pred_lr = log_reg.predict(X_test_s)
err = pd.DataFrame({"y_true": y_test, "y_pred": y_pred_lr}, index=test_idx)
for c in meta_cols:
    err[c] = df.loc[test_idx, c].values

fp = err[(err["y_true"] == 0) & (err["y_pred"] == 1)]
fn = err[(err["y_true"] == 1) & (err["y_pred"] == 0)]
print("False positives (sample up to 5):")
display(fp.head(5))
print("False negatives (sample up to 5):")
display(fn.head(5))


False positives (sample up to 5):


,y_true,y_pred,track_name,artist_name,genre,popularity
587,0,1,track_587,artist_27,hip hop,55
1000,0,1,track_1000,artist_40,hip hop,15
1187,0,1,track_1187,artist_67,rock,54
1598,0,1,track_1598,artist_78,pop,47
1283,0,1,track_1283,artist_3,rock,21


False negatives (sample up to 5):


,y_true,y_pred,track_name,artist_name,genre,popularity
2116,1,0,track_2116,artist_36,hip hop,59
551,1,0,track_551,artist_71,r&b,61
1773,1,0,track_1773,artist_13,r&b,67
1094,1,0,track_1094,artist_54,electronic,56
2226,1,0,track_2226,artist_66,hip hop,67


## 9. Conclusions

- Compare **ROC-AUC** and **F1** between models; the comparison table is saved to `outputs/tables/model_metrics.csv`.
- **Logistic coefficients** are on log-odds scale (per standardized feature unit). **RF importance** is split-based — rank correlation, not identical ranking.
- **Limitations:** popularity proxy; confounds; correlation is not causation.
